In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

2026-06-24 14:20:47.589679: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782310847.777037      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782310847.840040      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782310848.281414      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782310848.281470      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782310848.281474      58 computation_placer.cc:177] computation placer alr

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(

"/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification/Garbage classification",

validation_split=0.2,

subset="training",

seed=42,

image_size=(224,224),

batch_size=32

)

val_ds = tf.keras.utils.image_dataset_from_directory(

"/kaggle/input/datasets/asdasdasasdas/garbage-classification/Garbage classification/Garbage classification",

validation_split=0.2,

subset="validation",

seed=42,

image_size=(224,224),

batch_size=32

)

num_classes=len(train_ds.class_names)

Found 2527 files belonging to 6 classes.
Using 2022 files for training.


I0000 00:00:1782310925.356536      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1782310925.362497      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 2527 files belonging to 6 classes.
Using 505 files for validation.


In [4]:
base_model = tf.keras.applications.EfficientNetV2S(

weights="imagenet",

include_top=False,

input_shape=(224,224,3)

)

base_model.trainable=False

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [5]:
model=tf.keras.Sequential([

base_model,

tf.keras.layers.GlobalAveragePooling2D(),

tf.keras.layers.Dense(
512,
activation='relu'
),

tf.keras.layers.Dropout(
0.4
),

tf.keras.layers.Dense(
256,
activation='relu'
),

tf.keras.layers.Dropout(
0.3
),

tf.keras.layers.Dense(
num_classes,
activation='softmax'
)

])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetv2-s (Functional)   │ (None, 7, 7, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,120,102 (80.57 MB)

 Trainable params: 788,742 (3.01 MB)

 Non-trainable params: 20,331,360 (77.56 MB)

In [6]:
model.compile(

optimizer='adam',

loss='sparse_categorical_crossentropy',

metrics=['accuracy']

)

In [7]:
history=model.fit(

train_ds,

validation_data=val_ds,

epochs=20

)

Epoch 1/20


I0000 00:00:1782311019.774777     137 service.cc:152] XLA service 0x7c76bc003260 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1782311019.774831     137 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1782311019.774835     137 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1782311024.345292     137 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-06-24 14:23:53.635756: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:23:53.775019: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:23:54.667331: E external/local_xl

 1/64 ━━━━━━━━━━━━━━━━━━━━ 55:29 53s/step - accuracy: 0.1562 - loss: 1.8623

I0000 00:00:1782311051.533342     137 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


63/64 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.6300 - loss: 0.9883

2026-06-24 14:24:28.137370: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:24:28.271846: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:24:29.106648: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:24:29.243517: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:24:30.735449: E external/local_xla/xla/stream_

64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 536ms/step - accuracy: 0.6320 - loss: 0.9838

2026-06-24 14:25:01.829612: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:25:01.966340: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:25:02.828181: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:25:02.968118: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-06-24 14:25:04.459151: E external/local_xla/xla/stream_

64/64 ━━━━━━━━━━━━━━━━━━━━ 109s 885ms/step - accuracy: 0.7591 - loss: 0.6998 - val_accuracy: 0.8356 - val_loss: 0.4854
Epoch 2/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.8783 - loss: 0.3747 - val_accuracy: 0.8594 - val_loss: 0.4181
Epoch 3/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 109ms/step - accuracy: 0.8996 - loss: 0.2898 - val_accuracy: 0.8574 - val_loss: 0.4151
Epoch 4/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 112ms/step - accuracy: 0.9174 - loss: 0.2275 - val_accuracy: 0.8673 - val_loss: 0.4592
Epoch 5/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 114ms/step - accuracy: 0.9342 - loss: 0.1862 - val_accuracy: 0.8673 - val_loss: 0.4246
Epoch 6/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 7s 115ms/step - accuracy: 0.9397 - loss: 0.1694 - val_accuracy: 0.8772 - val_loss: 0.4869
Epoch 7/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 8s 119ms/step - accuracy: 0.9496 - loss: 0.1478 - val_accuracy: 0.8911 - val_loss: 0.4339
Epoch 8/20
64/64 ━━━━━━━━━━━━━━━━━━━━ 8s 123ms/step - accuracy: 0.9599 - loss: 0.1091 - val_accuracy: 0.8772 - va

In [8]:
loss,accuracy=model.evaluate(val_ds)

print(
"Accuracy:",
accuracy
)

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 95ms/step - accuracy: 0.8931 - loss: 0.5780
Accuracy: 0.893069326877594


In [9]:
from sklearn.metrics import (
precision_score,
recall_score,
f1_score
)

y_true=[]
y_pred=[]

for images,labels in val_ds:

    pred=model.predict(images)

    y_true.extend(labels.numpy())

    y_pred.extend(
        np.argmax(pred,axis=1)
    )

precision=precision_score(
y_true,
y_pred,
average='weighted'
)

recall=recall_score(
y_true,
y_pred,
average='weighted'
)

f1=f1_score(
y_true,
y_pred,
average='weighted'
)

print("Precision:",precision)

print("Recall:",recall)

print("F1:",f1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step
Precision: 0.8947734960651854
Recall: 0.8930693069306931
F1: 0.8924235127279317
